# Homework 6 — Decision Trees and Ensemble Learning

Predict `fuel_efficiency_mpg` with a decision tree, a random forest and XGBoost.

Run with **scikit-learn 1.7.2** and xgboost 3.1.1 (the versions the answer options were computed with). Newer scikit-learn (1.9) gives slightly different random-forest RMSEs and flips Q3 from 100 to 150.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

## Preparing the dataset

Fill missing values with zeros, split 60/20/20, one-hot encode with `DictVectorizer(sparse=True)`.

In [2]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.pop('fuel_efficiency_mpg').values
y_val = df_val.pop('fuel_efficiency_mpg').values
y_test = df_test.pop('fuel_efficiency_mpg').values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

## Question 1 — Decision tree with `max_depth=1`

In [3]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2 — Random forest, `n_estimators=10`

In [4]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
rmse = root_mean_squared_error(y_val, rf.predict(X_val))
round(rmse, 4)

1.8371

## Question 3 — Varying `n_estimators`

In [5]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, rf.predict(X_val))
    print(n, round(rmse, 3))

10 1.837


50 1.772


100 1.768


150 1.769


## Question 4 — Best `max_depth` by mean RMSE

In [6]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    print(d, round(np.mean(scores), 4))

10 1.7554


15 1.784


20 1.7882


25 1.7863


## Question 5 — Feature importance

In [7]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out()).sort_values(ascending=False)
print(imp.round(4))
imp[['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']].sort_values(ascending=False)

model_year                      0.2960
vehicle_weight                  0.2039
fuel_type=Gasoline              0.1607
horsepower                      0.0780
engine_displacement             0.0610
acceleration                    0.0552
origin=USA                      0.0421
drivetrain=Front-wheel drive    0.0201
num_doors                       0.0194
fuel_type=Diesel                0.0170
fuel_type=Hybrid                0.0158
drivetrain=All-wheel drive      0.0077
num_cylinders                   0.0073
origin=Asia                     0.0064
drivetrain=Rear-wheel drive     0.0058
origin=Europe                   0.0035
dtype: float64


vehicle_weight         0.203879
horsepower             0.078017
engine_displacement    0.061005
acceleration           0.055201
dtype: float64

## Question 6 — XGBoost, `eta` 0.3 vs 0.1

In [8]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

results = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    results[eta] = root_mean_squared_error(y_val, model.predict(dval))
results

{0.3: 1.8264579799904583, 0.1: 1.724810059827976}